In [1]:
pip install mlflow dagshub optuna xgboost imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
from google.colab import userdata
import os


os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

In [4]:

import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/")

In [5]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/cf10446d955b4f34a659da9fcaab7f50', creation_time=1783941190743, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1783941190743, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn
import optuna
from imblearn.combine import SMOTETomek

In [22]:
df = pd.read_csv("/content/yt_preprocessed_data.csv").dropna(subset=['Comment'])
df.shape

(17680, 3)

In [23]:
df.drop(columns=[ "Unnamed: 0"], inplace=True)

In [24]:
df.head()

,Comment,Sentiment
0,let not forget apple pay 2014 required brand n...,neutral
1,nz 50 retailer dont even contactless credit ca...,negative
2,forever acknowledge channel help lesson idea e...,positive
3,whenever go place doesnt take apple pay doesnt...,negative
4,apple pay convenient secure easy use used kore...,positive


In [25]:
df = df.dropna(subset=["Comment"])

df["Comment"] = df["Comment"].astype(str).str.strip()

df = df[df["Comment"] != ""]

In [26]:
df.head()

,Comment,Sentiment
0,let not forget apple pay 2014 required brand n...,neutral
1,nz 50 retailer dont even contactless credit ca...,negative
2,forever acknowledge channel help lesson idea e...,positive
3,whenever go place doesnt take apple pay doesnt...,negative
4,apple pay convenient secure easy use used kore...,positive


In [27]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df["Sentiment"] = df["Sentiment"].map({
    "neutral": 0,
    "positive": 1,
    "negative": 2
})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 2)  # biigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['Comment'])
y = df['Sentiment']

# step 4 imbalance hand
somot_tomek = SMOTETomek(random_state=42)
X_resampled, y_resampled = somot_tomek.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path=f"{model_name}_model",
            serialization_format="pickle"
        )


# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for LightGBM, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "LightGBM"
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for LightGBM
run_optuna_experiment()


[I 2026-07-13 11:57:28,025] A new study created in memory with name: no-name-b981260d-e509-44e9-80d3-461138953816


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.172236 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 77145
[LightGBM] [Info] Number of data points in the train set: 26412, number of used features: 999
[LightGBM] [Info] Start training from score -1.099976
[LightGBM] [Info] Start training from score -1.099408
[LightGBM] [Info] Start training from score -1.096457


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-13 11:57:34,164] Trial 0 finished with value: 0.5606542480690595 and parameters: {'n_estimators': 115, 'learning_rate': 0.0014258388528252959, 'max_depth': 7}. Best is trial 0 with value: 0.5606542480690595.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.246987 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 77145
[LightGBM] [Info] Number of data points in the train set: 26412, number of used features: 999
[LightGBM] [Info] Start training from score -1.099976
[LightGBM] [Info] Start training from score -1.099408
[LightGBM] [Info] Start training from score -1.096457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-13 11:57:37,094] Trial 1 finished with value: 0.4643343934575193 and parameters: {'n_estimators': 116, 'learning_rate': 0.0001461564115321866, 'max_depth': 3}. Best is trial 0 with value: 0.5606542480690595.


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.149216 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 77145
[LightGBM] [Info] Number of data points in the train set: 26412, number of used features: 999
[LightGBM] [Info] Start training from score -1.099976
[LightGBM] [Info] Start training from score -1.099408
[LightGBM] [Info] Start training from score -1.096457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-13 12:02:18,265] Trial 29 finished with value: 0.6784794790246857 and parameters: {'n_estimators': 275, 'learning_rate': 0.019636215768875576, 'max_depth': 7}. Best is trial 13 with value: 0.7575344540360442.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.160441 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 77145
[LightGBM] [Info] Number of data points in the train set: 26412, number of used features: 999
[LightGBM] [Info] Start training from score -1.099976
[LightGBM] [Info] Start training from score -1.099408
[LightGBM] [Info] Start training from score -1.096457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/13 12:02:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/13 12:02:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/4/runs/1c43c231b9184fcfac7527c327aa6fbf
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/4
